# On-Target Threshold
Determine the value of hyperparameter `cnfg.ON_TARGET_THRESHOLD`, which is used to decide if a gaze sample / fixation / visit is "on-target" or not.<br><br>

<i>TLDR:</i> We set `ON_TARGET_THRESHOLD = 1.75DVA`, which covers 100% of identified targets (hits) and 96% of cases when subjects performed an identification action (hits + false alarms). This threshold is also supported by the fixation analysis, where 95% of fixations that co-occurred with an identification action were within 1.22DVA from the target.

In [1]:

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from statsmodels.sandbox.stats.stats_dhuard import percentileofscore
import plotly.io as pio

import config as cnfg
from plgrnd2 import fixations

# pio.renderers.default = "notebook"
pio.renderers.default = "browser"

### Read data

In [2]:
from analysis.helpers.read_data import read_data
loaded_data = read_data(cnfg.OUTPUT_PATH)
idents = loaded_data.identifications
fixations = loaded_data.fixations
del loaded_data

In [3]:
DEFAULT_THRESHOLD = cnfg.ON_TARGET_THRESHOLD_DVA

### (A) Gaze on Target Identification
#### (1) Distances-from-Target when subject performed any identification
i.e. Hits or False-Alarms

In [4]:
percentiles = [0.5, 0.75, 0.85, 0.9, 0.95, 0.99,]

not_misses = idents.loc[idents["identification_category"] != "miss"]
dist_summary = (
    pd.concat([
        not_misses["distance_dva"].describe(percentiles).rename("all"),
        not_misses.groupby("subject")["distance_dva"].describe(percentiles).T,
    ], axis=1)
).T

if not_misses['distance_dva'].min() > DEFAULT_THRESHOLD:
    default_threshold_percentile = 0.0
elif not_misses['distance_dva'].max() < DEFAULT_THRESHOLD:
    default_threshold_percentile = 100.0
else:
    default_threshold_percentile = percentileofscore(not_misses['distance_dva'], DEFAULT_THRESHOLD)

print(f"When subjects identified a target, {dist_summary.loc['all', '95%']:.2f} DVA of distance covers 95% of the cases.")
print(f"The default threshold of {DEFAULT_THRESHOLD}DVA covers {default_threshold_percentile:.2f}% of the cases.")
dist_summary

When subjects identified a target, 1.46 DVA of distance covers 95% of the cases.
The default threshold of 1.75DVA covers 96.24% of the cases.


,count,mean,std,min,50%,75%,85%,90%,95%,99%,max
all,1248.0,0.827495,1.642685,0.005250,0.539752,0.840340,1.031440,1.187911,1.457244,8.401630,23.297429
2,112.0,0.887389,1.378940,0.005250,0.638451,0.945330,1.132480,1.240759,1.367437,8.195211,9.435844
12,111.0,1.377363,2.263406,0.160329,0.863651,1.245024,1.640418,1.947023,4.304638,9.813110,19.109826
13,99.0,0.782881,1.451985,0.073701,0.538058,0.731862,0.912429,1.155491,1.408013,6.124180,13.597367
14,104.0,1.307629,3.840592,0.032188,0.370250,0.633209,0.733201,0.991192,5.101486,21.035096,23.297429
15,98.0,0.809266,1.268136,0.085205,0.675779,0.865602,1.059637,1.202942,1.377850,2.032327,12.765376
16,90.0,0.423305,0.247078,0.043483,0.350909,0.640706,0.747622,0.778273,0.858737,0.915361,0.967981
17,101.0,0.764956,1.272487,0.065453,0.567267,0.825397,1.034370,1.153490,1.318819,3.755980,12.600789
18,123.0,0.442788,0.809888,0.057758,0.359878,0.493016,0.569625,0.628443,0.778915,0.995772,9.073654
19,116.0,0.792266,0.767282,0.108211,0.701126,1.014256,1.145919,1.215355,1.350801,1.971149,7.938850


In [5]:
fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True, shared_yaxes=False,
)

# top: distribution across all subjects
fig.add_trace(
    row=1, col=1, trace=go.Violin(
        y0="distance", x=not_misses["distance_dva"],
        name="All Subjects", legendgroup="All Subjects",
        text=not_misses.apply(
            lambda row: f"Subject: {row['subject']}<br>"
                        f"Trial: {row['trial']}<br>"
                        # f"Target: {row['target']}<br>"
                        f"Distance: {row['distance_dva']:.2f} DVA",
            axis=1
        ),
        marker=dict(color=cnfg.get_discrete_color("all")),
        width=1.75, orientation="h", side="positive", spanmode='hard',
        box=dict(visible=False),
        meanline=dict(visible=True),
        points="all", pointpos=-0.5,
        showlegend=True, hoverinfo="x+y+text",

    )
)

# bottom: distribution per subject
for subj_id in not_misses[cnfg.SUBJECT_STR].unique():
    subj_string = f"{cnfg.SUBJECT_STR.capitalize()} {subj_id:02d}"
    subj_data = not_misses[not_misses[cnfg.SUBJECT_STR] == subj_id]
    texts = subj_data.apply(
        lambda row: f"{subj_string}<br>"
                    f"Trial: {row['trial']}<br>"
                    # f"Target: {row['target']}<br>"
                    f"Distance: {row['distance_dva']:.2f} DVA",
        axis=1
    )
    fig.add_trace(
        row=2, col=1, trace=go.Violin(
            y0="distance", x=subj_data["distance_dva"],
            text=texts,
            name=subj_string, legendgroup=subj_string,
            marker=dict(color=cnfg.get_discrete_color(subj_id, loop=True), opacity=0.5),
            width=1.75, orientation="h", side="positive", spanmode='hard',
            box=dict(visible=False),
            meanline=dict(visible=True),
            points="all", pointpos=-0.5,
            showlegend=True, hoverinfo="x+y+text"
        )
    )

# update visuals
fig.update_annotations(font=cnfg.AXIS_LABEL_FONT)
fig.update_yaxes(showticklabels=False)  # Hide y-axis labels
fig.update_xaxes(
    title=None, showline=False,
    showgrid=True, gridcolor=cnfg.GRID_LINE_COLOR, gridwidth=cnfg.GRID_LINE_WIDTH,
    zeroline=False, zerolinecolor=cnfg.GRID_LINE_COLOR, zerolinewidth=cnfg.ZERO_LINE_WIDTH,
    tickfont=cnfg.AXIS_TICK_FONT,
)
fig.update_layout(
    width=1200, height=675,
    title=dict(text="Distance on Identification-Action", font=cnfg.TITLE_FONT),
    paper_bgcolor='rgba(0, 0, 0, 0)',
    # plot_bgcolor='rgba(0, 0, 0, 0)',
    showlegend=True,
)

fig.show()

##### (2) Distances-from-Target for `Hit`/`False Alarm` Identifications
We identify `hits` and `false-alarms` based on the distance of the gaze from the closest target when the subject performed an identification action. The threshold is set in `cnfg.ON_TARGET_THRESHOLD_DVA`.

In [6]:
hits = not_misses[not_misses["identification_category"] == "hit"]
hit_dist_summary = (
    pd.concat([
        hits["distance_dva"].describe(percentiles).rename("all"),
        hits.groupby("subject")["distance_dva"].describe(percentiles).T,
    ], axis=1)
).T

if hits['distance_dva'].min() > DEFAULT_THRESHOLD:
    default_threshold_percentile = 0.0
elif hits['distance_dva'].max() < DEFAULT_THRESHOLD:
    default_threshold_percentile = 100.0
else:
    default_threshold_percentile = percentileofscore(hits['distance_dva'], DEFAULT_THRESHOLD)

print(f"For identifications classified as `hits`, {hit_dist_summary.loc['all', '95%']:.2f} DVA of distance covers 95% of the cases.")
print(f"The default threshold of {DEFAULT_THRESHOLD}DVA covers {default_threshold_percentile:.2f}% of the cases.")
hit_dist_summary

For identifications classified as `hits`, 1.24 DVA of distance covers 95% of the cases.
The default threshold of 1.75DVA covers 100.00% of the cases.


,count,mean,std,min,50%,75%,85%,90%,95%,99%,max
all,1191.0,0.583955,0.342398,0.005250,0.522377,0.802773,0.952180,1.072676,1.238744,1.518726,1.712458
2,108.0,0.641263,0.366159,0.005250,0.592702,0.936236,1.064274,1.187911,1.260981,1.412605,1.438825
12,92.0,0.791264,0.363523,0.160329,0.779886,1.006058,1.158340,1.294181,1.469663,1.649469,1.680471
13,95.0,0.569296,0.300453,0.073701,0.525737,0.706520,0.833511,0.952872,1.193932,1.403128,1.517925
14,95.0,0.409261,0.268807,0.032188,0.350813,0.602102,0.646063,0.667448,0.955954,1.180261,1.207467
15,96.0,0.688470,0.348134,0.085205,0.675779,0.859317,1.031871,1.180632,1.333408,1.465944,1.700377
16,90.0,0.423305,0.247078,0.043483,0.350909,0.640706,0.747622,0.778273,0.858737,0.915361,0.967981
17,98.0,0.602429,0.306102,0.065453,0.565192,0.781139,0.915732,1.062111,1.173186,1.364848,1.398881
18,122.0,0.372043,0.201640,0.057758,0.359751,0.491214,0.568692,0.615991,0.756176,0.939437,1.009208
19,110.0,0.709812,0.346265,0.108211,0.677313,0.994107,1.094975,1.177265,1.301972,1.445614,1.525928


In [7]:
fas = not_misses[not_misses["identification_category"] == "false_alarm"]
fas_dist_summary = (
    pd.concat([
        fas["distance_dva"].describe(percentiles).rename("all"),
        fas.groupby("subject")["distance_dva"].describe(percentiles).T,
    ], axis=1)
).T

if fas['distance_dva'].min() > DEFAULT_THRESHOLD:
    default_threshold_percentile = 0.0
elif fas['distance_dva'].max() < DEFAULT_THRESHOLD:
    default_threshold_percentile = 100.0
else:
    default_threshold_percentile = percentileofscore(fas['distance_dva'], DEFAULT_THRESHOLD)

print(f"For identifications classified as `false alarms`, {fas_dist_summary.loc['all', '95%']:.2f} DVA of distance covers 95% of the cases.")
print(f"The default threshold of {DEFAULT_THRESHOLD}DVA covers {default_threshold_percentile:.2f}% of the cases.")
fas_dist_summary

For identifications classified as `false alarms`, 18.66 DVA of distance covers 95% of the cases.
The default threshold of 1.75DVA covers 0.00% of the cases.


,count,mean,std,min,50%,75%,85%,90%,95%,99%,max
all,47.0,7.009501,5.432442,1.785373,5.971665,8.817337,12.617248,14.348471,18.656638,22.305638,23.297429
2,4.0,7.532805,2.123616,4.494146,8.100615,8.525380,8.889565,9.071658,9.253751,9.399425,9.435844
12,15.0,5.051693,4.757665,1.816095,2.261394,6.712295,8.436063,9.395747,12.699509,17.827762,19.109826
13,4.0,5.855520,5.503440,1.785373,4.019670,7.878091,10.165802,11.309657,12.453512,13.368596,13.597367
14,7.0,13.732780,7.678483,3.594204,15.475127,19.370280,21.356967,22.003788,22.650608,23.168065,23.297429
15,1.0,12.765376,NaN,12.765376,12.765376,12.765376,12.765376,12.765376,12.765376,12.765376,12.765376
17,3.0,6.074165,5.730696,1.865727,3.755980,8.178384,9.947346,10.831827,11.716308,12.423893,12.600789
18,1.0,9.073654,NaN,9.073654,9.073654,9.073654,9.073654,9.073654,9.073654,9.073654,9.073654
19,3.0,3.932055,3.470517,1.867956,1.989360,4.964105,6.154003,6.748952,7.343901,7.819860,7.938850
20,2.0,6.127572,0.145192,6.024906,6.127572,6.178905,6.199439,6.209705,6.219972,6.228185,6.230239


### (B) Fixation Analysis
#### (3) Distances-from-Target across all fixations

In [8]:
percentiles = [0.05, 0.25, 0.5, 0.75, 0.9, 0.95]

dva_cols = [col for col in fixations.columns if col.endswith("distance_dva")]
min_dists = pd.concat([fixations[["subject", "trial", "eye", "event"]], fixations[dva_cols].min(axis=1).rename("distance")], axis=1)
fixation_dist_summary = (
    pd.concat([
        min_dists["distance"].describe(percentiles).rename("all"),
        min_dists.groupby("subject")["distance"].describe(percentiles).T,
    ], axis=1)
).T

print("All Fixations:")
fixation_dist_summary

All Fixations:


,count,mean,std,min,5%,25%,50%,75%,90%,95%,max
all,59047.0,8.486464,5.238046,0.006736,0.820594,4.512373,8.040041,11.771160,15.347914,17.860353,43.422000
2,6154.0,7.849844,4.980788,0.028776,0.657545,3.976750,7.520965,10.975553,14.343740,16.802477,28.950040
12,6283.0,8.633137,5.309519,0.028722,1.040453,4.685636,8.050677,11.787886,15.376574,18.479427,31.039929
13,4102.0,8.578085,5.543238,0.041144,0.709186,4.476645,8.051332,11.780995,15.823263,18.768281,32.165304
14,4785.0,9.000730,5.330353,0.042179,0.829029,4.934618,8.498420,12.582696,16.174535,18.365145,43.422000
15,3808.0,8.276203,5.080044,0.062877,0.792611,4.287803,8.028084,11.589419,15.156872,17.100512,39.595361
16,5014.0,8.777813,4.987164,0.035050,0.981601,4.987913,8.463345,12.054985,15.480871,17.715331,27.668284
17,3863.0,8.085943,5.135572,0.115526,0.940019,4.116761,7.564736,11.372057,14.914327,17.229710,29.743946
18,5707.0,8.284476,5.163665,0.014692,0.608860,4.477136,7.674629,11.428806,15.112763,17.852830,29.960784
19,3647.0,7.997410,5.370108,0.022844,0.724185,3.708774,7.639254,11.345961,14.595699,17.316863,31.603614


#### (4) Distances-from-Target during identification-fixations
##### find identification fixations
fixations where either:
- the subject performed an identification action during the fixation
- the subject performed an identification action immediately after the fixation

In [9]:
fixs_with_ident_time = fixations.copy()
fixs_with_ident_time["target"] = fixs_with_ident_time[dva_cols].idxmin(axis=1).str.replace("_distance_dva", "")
fixs_with_ident_time["distance_dva"] = fixs_with_ident_time[dva_cols].min(axis=1)
fixs_with_ident_time = (
    fixs_with_ident_time
    .drop(columns=[col for col in fixs_with_ident_time.columns if "_distance_" in col])
    .merge(
        idents.loc[
            idents["identification_category"] == "hit", ["subject", "trial", "target", "time"]
        ], on=["subject", "trial", "target"], how="left"
    )
)

fixs_with_ident_time.loc[:, "is_during"] = (fixs_with_ident_time["start_time"] <= fixs_with_ident_time["time"]) & (fixs_with_ident_time["time"] <= fixs_with_ident_time["end_time"])

fixs_with_ident_time.loc[:, "end_to_ident_diff"] = fixs_with_ident_time["time"] - fixs_with_ident_time["end_time"]
fixs_with_ident_time.loc[:, "is_immediately_preceding"] = False
immediately_preceding_idxs = (
    fixs_with_ident_time
    .loc[(0 <= fixs_with_ident_time["end_to_ident_diff"]) & (fixs_with_ident_time["end_to_ident_diff"] <= 1000)]    # max 1 sec
    .groupby(["subject", "trial", "eye", "target"])["end_to_ident_diff"]
    .idxmin()
    .values
)
fixs_with_ident_time.loc[immediately_preceding_idxs, "is_immediately_preceding"] = True
# fixs_with_ident_time.drop(columns=["end_to_ident_diff"], inplace=True)

In [10]:
ident_fixs = fixs_with_ident_time.loc[fixs_with_ident_time["is_during"]]
ident_fixs_dist_summary = (
    pd.concat([
        ident_fixs["distance_dva"].describe(percentiles).rename("all"),
        ident_fixs.groupby("subject")["distance_dva"].describe(percentiles).T,
    ], axis=1)
).T

print("Identification Fixations:")
print(f"When subjects fixated on a target and identified it during the fixation, {ident_fixs_dist_summary.loc['all', '95%']:.2f} DVA of distance covers 95% of the cases.")
ident_fixs_dist_summary

Identification Fixations:
When subjects fixated on a target and identified it during the fixation, 1.22 DVA of distance covers 95% of the cases.


,count,mean,std,min,5%,25%,50%,75%,90%,95%,max
all,1167.0,0.570264,0.335903,0.006736,0.125234,0.326753,0.498505,0.779355,1.049575,1.220444,1.711117
2,100.0,0.622015,0.355421,0.038549,0.137867,0.316907,0.558603,0.907463,1.082942,1.188417,1.484959
12,91.0,0.778684,0.352282,0.200029,0.319005,0.485924,0.753589,0.999507,1.234216,1.428907,1.630773
13,94.0,0.558882,0.305073,0.041144,0.160992,0.346736,0.499201,0.754279,0.952528,1.120638,1.499542
14,94.0,0.408975,0.244394,0.042179,0.083710,0.235990,0.379883,0.530890,0.724086,0.856253,1.232382
15,95.0,0.696551,0.347877,0.062877,0.201703,0.420215,0.708395,0.889473,1.163865,1.351245,1.711117
16,90.0,0.387415,0.210815,0.039001,0.082544,0.235339,0.343308,0.584196,0.698830,0.725568,0.795743
17,96.0,0.604510,0.298902,0.115526,0.199678,0.397338,0.540568,0.799723,1.051890,1.206110,1.372795
18,118.0,0.363074,0.203199,0.014692,0.080681,0.192735,0.343978,0.497836,0.595048,0.727756,0.923605
19,109.0,0.685320,0.353512,0.022844,0.183434,0.392898,0.659171,0.974525,1.129386,1.257403,1.536005


#### (5) Distances-from-Target during pre-identification-fixations
Distance from target for fixations that immediately precede an identification fixation.

In [11]:
preceding_fixs = fixs_with_ident_time.loc[fixs_with_ident_time["is_immediately_preceding"]]
preceding_fixs_dist_summary = (
    pd.concat([
        preceding_fixs["distance_dva"].describe(percentiles).rename("all"),
        preceding_fixs.groupby("subject")["distance_dva"].describe(percentiles).T,
    ], axis=1)
).T

print("Preceding Identification Fixations:")
print(f"When subjects fixated on a target and identified it immediately after the fixation, {preceding_fixs_dist_summary.loc['all', '95%']:.2f} DVA of distance covers 95% of the cases.")
preceding_fixs_dist_summary

Preceding Identification Fixations:
When subjects fixated on a target and identified it immediately after the fixation, 4.08 DVA of distance covers 95% of the cases.


,count,mean,std,min,5%,25%,50%,75%,90%,95%,max
all,1153.0,1.351999,1.835600,0.028722,0.212420,0.526404,0.895274,1.434105,2.378837,4.084467,19.686305
2,103.0,1.717825,2.093684,0.070308,0.198982,0.547017,0.979276,1.992934,4.471567,5.633688,11.768530
12,90.0,1.690035,2.156922,0.028722,0.345664,0.716014,1.152576,1.609463,2.505978,5.689507,15.175831
13,93.0,0.980775,0.753262,0.057448,0.239623,0.548239,0.776078,1.238520,1.628010,2.263543,5.284142
14,90.0,1.613251,2.500898,0.116917,0.260698,0.527101,0.880610,1.646037,3.367300,5.009005,16.502108
15,93.0,1.988552,3.135193,0.076795,0.202344,0.600615,0.981594,1.909499,3.817916,8.067126,16.923690
16,86.0,1.507019,2.337281,0.092160,0.198212,0.578251,0.926373,1.422448,2.784769,4.767453,19.686305
17,94.0,1.799421,1.938471,0.121700,0.326927,0.735199,1.296061,1.773725,4.329056,6.180865,9.824945
18,117.0,0.826970,0.767981,0.036474,0.126664,0.356095,0.591683,0.943717,1.695956,2.588743,4.105434
19,105.0,1.342906,1.585665,0.128752,0.269251,0.666644,1.024923,1.518003,1.856176,3.712830,13.897471


### Visualize

In [12]:
column_titles = ["All Fixations", "Co-Occurring with Identification", "Preceding Identification"]
fig = make_subplots(
    rows=2, cols=len(column_titles), column_titles=column_titles,
    shared_xaxes=True, shared_yaxes=False,
)

for c in range(len(column_titles)):
    if c == 0:
        data = fixs_with_ident_time
    elif c == 1:
        data = fixs_with_ident_time[fixs_with_ident_time["is_during"]]
    elif c == 2:
        data = fixs_with_ident_time[fixs_with_ident_time["is_immediately_preceding"]]
    else:
        raise ValueError(f"Unexpected column index {c}.")
    # top: distribution across all subjects
    fig.add_trace(
        row=1, col=c+1, trace=go.Violin(
            y0="distance", x=data["distance_dva"],
            name="All Subjects", legendgroup="All Subjects",
            text=data.apply(
                lambda row: f"Subject: {row['subject']}<br>"
                            f"Trial: {row['trial']}<br>"
                            f"Target: {row['target']}<br>"
                            f"Distance: {row['distance_dva']:.2f} DVA",
                axis=1
            ),
            marker=dict(color=cnfg.get_discrete_color("all")),
            width=1.75, orientation="h", side="positive", spanmode='hard',
            box=dict(visible=False),
            meanline=dict(visible=True),
            points="all", pointpos=-0.5,
            showlegend=c==0, hoverinfo="x+y+text",

        )
    )
    # bottom: distribution per subject
    for subj_id in data[cnfg.SUBJECT_STR].unique():
        subj_string = f"{cnfg.SUBJECT_STR.capitalize()} {subj_id:02d}"
        subj_data = data[data[cnfg.SUBJECT_STR] == subj_id]
        texts = subj_data.apply(
            lambda row: f"{subj_string}<br>"
                        f"Trial: {row['trial']}<br>"
                        f"Target: {row['target']}<br>"
                        f"Distance: {row['distance_dva']:.2f} DVA",
            axis=1
        )
        fig.add_trace(
            row=2, col=c+1, trace=go.Violin(
                y0="distance", x=subj_data["distance_dva"],
                text=texts,
                name=subj_string, legendgroup=subj_string,
                marker=dict(color=cnfg.get_discrete_color(subj_id, loop=True), opacity=0.5),
                width=1.75, orientation="h", side="positive", spanmode='hard',
                box=dict(visible=False),
                meanline=dict(visible=True),
                points="all", pointpos=-0.5,
                showlegend=c==0, hoverinfo="x+y+text"
            )
        )

# update visuals
fig.update_annotations(font=cnfg.AXIS_LABEL_FONT)
fig.update_yaxes(showticklabels=False)  # Hide y-axis labels
fig.update_xaxes(
    title=None, showline=False,
    showgrid=True, gridcolor=cnfg.GRID_LINE_COLOR, gridwidth=cnfg.GRID_LINE_WIDTH,
    zeroline=False, zerolinecolor=cnfg.GRID_LINE_COLOR, zerolinewidth=cnfg.ZERO_LINE_WIDTH,
    tickfont=cnfg.AXIS_TICK_FONT,
)
fig.update_layout(
    width=1400, height=650,
    title=dict(text="Distance on Fixations", font=cnfg.TITLE_FONT),
    paper_bgcolor='rgba(0, 0, 0, 0)',
    # plot_bgcolor='rgba(0, 0, 0, 0)',
    showlegend=True,
)

fig.show()